# 📓 Código Sintético — Trazabilidad y Logs Estructurados

Implementación completa para ejecutar en Google Colab.

**Aviso importante sobre esta versión del notebook:** la versión anterior pegaba el código completo del logger y del agente ReAct directamente en las celdas (una copia congelada, y con bugs ya corregidos en el paquete real — por ejemplo, enmascaraba `input_tokens`/`output_tokens` como si fueran secretos por contener la palabra "token"). Esta versión, en cambio, **instala el paquete `trazabilidad` real del repositorio** y lo importa, así que siempre estás usando el código corregido y probado con tests.

## 1. Clonar el repositorio e instalar el paquete

In [ ]:
REPO_URL = "https://github.com/<tu-usuario>/<tu-repo>.git"  # <-- edita esto

import os
if not os.path.isdir("codigo_sintetico"):
    get_ipython().system(f"git clone --depth 1 {REPO_URL} codigo_sintetico")
get_ipython().run_line_magic("cd", "codigo_sintetico")
get_ipython().system('pip install -e ".[all]" -q')
print("✅ Paquetes sintetico y trazabilidad instalados")

## 2. Configuración de API key (opcional)

Sin ninguna key, la demo usa un simulador ReAct determinista y lo indica explícitamente.

In [ ]:
import os
from getpass import getpass

# Descomenta si tienes API key real:
# os.environ["ANTHROPIC_API_KEY"] = getpass("Introduce tu API key de Anthropic: ")

## 3. Importar el logger estructurado y el agente ReAct

Ya no se define ninguna clase aquí: se importa directamente del paquete instalado.

In [ ]:
from trazabilidad import StructuredAgentLogger, ReActAgentWithLogging, list_files, read_file, search_docs

print("✅ Importado desde el paquete trazabilidad (no copiado/pegado)")

## 4. Configurar un LLM real (si configuraste una key) o un simulador

In [ ]:
import os

if os.environ.get("ANTHROPIC_API_KEY"):
    from sintetico.real_providers import create_provider
    from sintetico.providers import LLMRequest

    provider = create_provider("anthropic", default_model="haiku")

    def llm_call(prompt: str, params: dict) -> dict:
        response = provider.complete(LLMRequest(
            system_prompt="", messages=[{"role": "user", "content": prompt}],
            max_tokens=params.get("max_tokens", 512), temperature=params.get("temperature", 0.2),
        ))
        tokens = response.usage.get("input_tokens", 0) + response.usage.get("output_tokens", 0)
        return {"content": response.content, "tokens": tokens}

    print("✅ Usando Anthropic real (modelo: haiku)")
else:
    def llm_call(prompt: str, params: dict) -> dict:
        lowered = prompt.lower()
        if "observation:" not in lowered:
            content = ('Thought: Necesito listar los archivos disponibles.\n'
                       'Action: list_files\nAction Input: {"directory": "."}')
        else:
            content = ('Thought: Ya tengo suficiente información.\n'
                       'Action: None\nAction Input: {"result": "Tarea completada."}')
        return {"content": content, "tokens": 40}

    print("📝 Sin API key: usando un simulador ReAct determinista")

## 5. Demostración completa: agente + logging estructurado

In [ ]:
logger = StructuredAgentLogger(agent_id="demo-agent-colab", team_id="colab-notebook", log_level="INFO")

agent = ReActAgentWithLogging(
    llm_call=llm_call,
    tools={"list_files": list_files, "read_file": read_file, "search_docs": search_docs},
    logger=logger, max_cycles=5,
)

result = agent.run(user_query="¿Qué archivos hay disponibles en el proyecto?")

print("\n--- Resultado ---")
print(result)

## 6. Ver las trazas en el dashboard (opcional)

El repositorio incluye una API + dashboard de observabilidad (`sintetico_api`) que visualiza exactamente este tipo de trazas — con costes, latencias y una vista tipo "waterfall" de cada ejecución, al estilo Datadog/CloudWatch. Consulta el notebook principal (`notebooks/codigo_sintetico_colab.ipynb`), sección 5, para arrancarlo desde Colab, o localmente:

```bash
pip install -e ".[api]"
uvicorn sintetico_api.main:app --reload
# http://localhost:8000/
```